In [38]:
import numpy as np
from scipy.linalg import svd

def projection_splitting(A, rank, max_iter=100, tol=1e-6):
    m, n = A.shape
    
    # Initialize with orthogonal matrices
    U, _ = np.linalg.qr(np.random.randn(m, rank))  # Fix 1: Ensure orthonormal columns
    V, _ = np.linalg.qr(np.random.randn(n, rank))
    
    for k in range(max_iter):
        # --- Column Space Update ---
        # Compute left singular vectors of A @ V (shape m x rank)
        U_svd, _, _ = svd(A @ V, full_matrices=False)
        U_new = U_svd[:, :rank]  # Fix 2: Take left singular vectors
        
        # --- Row Space Update ---
        # Compute left singular vectors of A.T @ U_new (shape n x rank)
        V_svd, _, _ = svd(A.T @ U_new, full_matrices=False)  # Fix 3: Use updated U_new
        V_new = V_svd[:, :rank]
        
        # --- Check Convergence ---
        X_old = U @ V.T
        X_new = U_new @ V_new.T
        if np.linalg.norm(X_new - X_old, 'fro') < tol:  # Fix 4: Compare full matrices
            print(f"Converged in {k+1} iterations.")
            break
            
        U, V = U_new, V_new
    
    return U @ V.T, U, V


In [40]:
m, n, r = 100, 50, 5
U_true = np.random.randn(m, r)
V_true = np.random.randn(n, r)
A = U_true @ V_true.T  # Exact rank-r matrix

X, U, V = projection_splitting(A, rank=r, max_iter=1000)
error = np.linalg.norm(A - X, 'fro')
print(f"Reconstruction error: {error:.2e}")  # Should be ~1e-15

Converged in 3 iterations.
Reconstruction error: 1.59e+02


In [16]:
def test_exact_low_rank():
    m, n, r = 100, 50, 5
    U_true = np.random.randn(m, r)
    V_true = np.random.randn(n, r)
    A = U_true @ V_true.T  # Exact rank-r matrix
    
    X, U, V = projection_splitting(A, rank=r)
    
    # Check if ||A - X||_F is small
    error = np.linalg.norm(A - X, 'fro')
    print(f"Exact low-rank test error: {error:.2e}")
    assert error < 1e-6, "Failed on exact low-rank test!"

In [17]:
test_exact_low_rank()

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 5 is different from 100)

In [14]:
r = 3
X_approx, iterations = alternating_projections(A, r, epsilon=1e-6, max_iter=1000, init_method='svd')
X_approx


array([[ 0.32606428, -0.28643834, -0.60872755, -0.05212348, -0.19661956,
         0.07344355,  0.48733369],
       [-0.04385069,  0.2501597 , -0.36717543, -0.04394952,  0.72017119,
        -0.05381578,  0.33027822],
       [ 0.16888498, -0.2631184 , -0.11081327,  0.00280241, -0.44205989,
         0.0667468 ,  0.06851209],
       [ 0.12769259,  0.11906915,  0.06703016, -0.12033178, -0.05251953,
        -0.1188673 , -0.00365757],
       [ 0.0136026 ,  0.03768921, -0.0211486 , -0.02021566,  0.05355029,
        -0.02095307,  0.02687077],
       [-0.40327864, -0.32623379, -0.08168024,  0.35493965,  0.11195968,
         0.33556958, -0.08093185],
       [-0.32531599, -0.30318322, -0.12691883,  0.30406774,  0.09361196,
         0.29726746, -0.02518688]])